In [3]:
# Cell 1
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)


SEED_DIGIT = 5      
ID_SUM     = 26      
LAST_NAME  = "Lancion"   
STUDENT_ID = "TUPM-26-1845"   
DATA_PATH   = "data/secom.data"
LABELS_PATH = "data/secom_labels.data"
MAX_MISSING_PCT = 50.0  
OUT_DIR = "ex2_pass_fail_profiling"   
# ------------------------------------------------------
print("Student:", LAST_NAME, "| ID:", STUDENT_ID)
print("SEED_DIGIT =", SEED_DIGIT, "| ID_SUM =", ID_SUM)
X_df = pd.read_csv(DATA_PATH, sep=r"\s+", header=None).apply(pd.to_numeric, errors="coerce")
y = pd.read_csv(LABELS_PATH, sep=r"\s+", header=None, usecols=[0], names=["label"])["label"].to_numpy()
X_df.index.name = "record_id"          # original 0-based record identifier
X = X_df.to_numpy(dtype=float)
assert X.shape[0] == len(y), "data / label row mismatch"
fname = lambda i: f"F{i:03d}"

n_obs = (~np.isnan(X)).sum(axis=0)
missing_pct = np.isnan(X).mean(axis=0) * 100
constant = (n_obs == 0) | (np.nanmin(X, axis=0) == np.nanmax(X, axis=0))
usable_idx = np.where(~constant & (missing_pct <= MAX_MISSING_PCT))[0]
print("Matrix:", X.shape, "| Pass(-1):", (y == -1).sum(), "| Fail(1):", (y == 1).sum())
print("Features with missing:", (missing_pct > 0).sum(), "| constant/empty:", constant.sum(), "| usable:", len(usable_idx))
os.makedirs(OUT_DIR, exist_ok=True)

Student: Lancion | ID: TUPM-26-1845
SEED_DIGIT = 5 | ID_SUM = 26
Matrix: (1567, 590) | Pass(-1): 1463 | Fail(1): 104
Features with missing: 538 | constant/empty: 116 | usable: 446


In [4]:
# Cell 2

rng2 = np.random.default_rng(SEED_DIGIT)
sel2 = np.sort(rng2.choice(usable_idx, size=5, replace=False))     # Step 3 selection
cols2 = [fname(i) for i in sel2]
df = pd.DataFrame(X[:, sel2], columns=cols2, index=X_df.index)
df["outcome"] = np.where(y == 1, "Fail", "Pass")
print("Selected:", cols2)


full_med = np.nanmedian(X[:, sel2], axis=0)
df[cols2] = df[cols2].fillna(dict(zip(cols2, full_med)))
print("Remaining NaN:", int(df[cols2].isna().sum().sum()))


n_pass, n_fail = (df.outcome == "Pass").sum(), (df.outcome == "Fail").sum()
n_per = min(20 + ID_SUM, n_pass, n_fail)
balanced = (pd.concat([df[df.outcome == g].sample(n=n_per, random_state=SEED_DIGIT) for g in ["Pass", "Fail"]])
              .sort_index())
balanced.to_csv(os.path.join(OUT_DIR, "balanced_sample.csv"))
print("Pass:", n_pass, "| Fail:", n_fail, "| per outcome:", n_per, "| total:", len(balanced))


grp = balanced.groupby("outcome")[cols2].agg(["count", "mean", "median", "std"])
display(grp)

rows = []
for c in cols2:
    p, f = grp.loc["Pass", c], grp.loc["Fail", c]
    n1, n2 = p["count"], f["count"]
    pooled = np.sqrt(((n1 - 1) * p["std"]**2 + (n2 - 1) * f["std"]**2) / (n1 + n2 - 2))
    amd = abs(f["mean"] - p["mean"])
    smd = amd / pooled if pooled > 0 else 0.0     # safe zero denominator
    rows.append({"feature": c, "n_pass": n1, "n_fail": n2,
                 "mean_pass": p["mean"], "mean_fail": f["mean"],
                 "median_pass": p["median"], "median_fail": f["median"],
                 "std_pass": p["std"], "std_fail": f["std"],
                 "abs_mean_diff": amd, "pooled_sd": pooled, "abs_smd": abs(smd)})
comp = pd.DataFrame(rows).sort_values("abs_smd", ascending=False).reset_index(drop=True)
comp.insert(0, "rank", comp.index + 1)
comp.to_csv(os.path.join(OUT_DIR, "feature_comparison.csv"), index=False)
display(comp)


ok = True                                          # NumPy cross-check of pandas stats
for c in cols2:
    a = balanced.loc[balanced.outcome == "Pass", c].to_numpy()
    b = balanced.loc[balanced.outcome == "Fail", c].to_numpy()
    r = comp.set_index("feature").loc[c]
    ok &= np.allclose([a.mean(), b.mean(), np.median(a), np.median(b), a.std(ddof=1), b.std(ddof=1)],
                      [r.mean_pass, r.mean_fail, r.median_pass, r.median_fail, r.std_pass, r.std_fail])
print("Balanced Sample Size            :", len(balanced), f"({n_per} per outcome)")
print("Selected Features               :", cols2)
print("Highest-Ranked Feature          :", comp.loc[0, "feature"])
print("Absolute Standardized Mean Diff :", round(comp.loc[0, "abs_smd"], 6))
print("NumPy-Pandas Statistics Match   :", bool(ok))

Selected: ['F011', 'F253', 'F376', 'F460', 'F469']
Remaining NaN: 0
Pass: 1463 | Fail: 104 | per outcome: 46 | total: 92


F011                              F253                               \
        count      mean  median       std count      mean   median       std   
outcome                                                                        
Fail       46  0.963415  0.9638  0.009401    46  0.035585  0.02990  0.035055   
Pass       46  0.964050  0.9659  0.008169    46  0.034691  0.03075  0.015181   

         F376                               F460                      \
        count      mean   median       std count       mean   median   
outcome                                                                
Fail       46  0.001676  0.00145  0.000767    46  34.254452  32.2964   
Pass       46  0.001570  0.00150  0.000504    46  26.168565  23.7800   

                    F469                               
               std count      mean   median       std  
outcome                                                
Fail     14.583294    46  6.205602  5.63645  4.292203  
Pass     11.307428    46  5.682807  5.62950  1.007554

,rank,feature,n_pass,n_fail,mean_pass,mean_fail,median_pass,median_fail,std_pass,std_fail,abs_mean_diff,pooled_sd,abs_smd
0,1,F460,46.0,46.0,26.168565,34.254452,23.78000,32.29640,11.307428,14.583294,8.085887,13.048571,0.619676
1,2,F469,46.0,46.0,5.682807,6.205602,5.62950,5.63645,1.007554,4.292203,0.522796,3.117545,0.167695
2,3,F376,46.0,46.0,0.001570,0.001676,0.00150,0.00145,0.000504,0.000767,0.000107,0.000649,0.164147
3,4,F011,46.0,46.0,0.964050,0.963415,0.96590,0.96380,0.008169,0.009401,0.000635,0.008807,0.072079
4,5,F253,46.0,46.0,0.034691,0.035585,0.03075,0.02990,0.015181,0.035055,0.000893,0.027012,0.033077


Balanced Sample Size            : 92 (46 per outcome)
Selected Features               : ['F011', 'F253', 'F376', 'F460', 'F469']
Highest-Ranked Feature          : F460
Absolute Standardized Mean Diff : 0.619676
NumPy-Pandas Statistics Match   : True
